# RetailPulse: Phase 3 – Customer Segmentation Engine (F1)

## 1. Overview & Business Rationale
In Phase 3, we implement two complementary approaches to customer segmentation:
1. **Heuristic RFM Scoring**: Quintile-based scoring (1–5) and rule-based segment labeling (Champions, Loyal, At Risk, Lost, etc.).
2. **Unsupervised Machine Learning**: K-Means clustering on log1p-transformed, standardized RFM features.
3. **Model Selection**: Elbow method (Inertia) and Silhouette score maximization.
4. **Cross-Comparison**: Alignment matrix contrasting heuristic segments against data-driven KMeans clusters.
5. **MLOps Governance**: Full parameter, metric, and artifact tracking in MLflow.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import json

root_dir = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(root_dir))

from src.utils.config import load_config
from src.models.segmentation import assign_rfm_quintiles, find_optimal_k, train_customer_segmentation

cfg = load_config(str(root_dir / 'config.yaml'))
print("Config loaded successfully.")

## 2. Load Customer Features

In [ ]:
rfm_path = root_dir / 'data' / 'processed' / 'customer_features.parquet'
df_rfm = pd.read_parquet(rfm_path)
print(f"Loaded {len(df_rfm):,} customer profiles.")
df_rfm.head()

## 3. Heuristic RFM Scoring & Persona Mapping

In [ ]:
df_scored = assign_rfm_quintiles(df_rfm)
print("RFM Segment Distribution:")
display(df_scored['RFM_Segment'].value_counts().to_frame())

## 4. Run Unsupervised KMeans Clustering & Optimization

In [ ]:
df_seg, kmeans, scaler, pca, metrics = train_customer_segmentation(
    df_rfm,
    features=['Recency', 'Frequency', 'Monetary'],
    log_to_mlflow=True
)

print(f"Selected Optimal k: {metrics['k']}")
print(f"Silhouette Score: {metrics['silhouette_score']:.4f}")
print(f"Davies-Bouldin Score: {metrics['davies_bouldin_score']:.4f}")
print(f"PCA Variance Explained: {sum(metrics['pca_variance_ratio'])*100:.1f}%")

## 5. Cross-Tabulation: Heuristic Segments vs Machine Learning Clusters

In [ ]:
cross_tab = pd.crosstab(df_seg['RFM_Segment'], df_seg['KMeans_Cluster'], margins=True)
display(cross_tab)

## 6. Cluster Archetype Profiles

In [ ]:
pd.DataFrame(metrics['cluster_profiles'])